라이브러리 설치 및 임포트

In [1]:
!pip install koreanize-matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 48.7 MB/s eta 0:00:00


In [2]:
# 라이브러리 설치
!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost scipy catboost

import pandas as pd
import numpy as np
import koreanize_matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


In [3]:
# Google Drive & 데이터 로드

from google.colab import drive
drive.mount('/content/drive')

print("\n" + "=" * 80)
print("📊 데이터 로드")
print("=" * 80)

train = pd.read_csv('train.csv')  # 업로드한 경우
test = pd.read_csv('test.csv')

print(f"✓ 훈련 데이터: {train.shape}")
print(f"✓ 테스트 데이터: {test.shape}")

# 데이터 준비
y_train = train['임신 성공 여부'].copy()
X_train = train.drop(['ID', '임신 성공 여부'], axis=1)
X_test = test.drop('ID', axis=1)

print(f"\n타겟 분포:")
print(f"  실패 (0): {(y_train==0).sum():,}")
print(f"  성공 (1): {(y_train==1).sum():,}")

Mounted at /content/drive

📊 데이터 로드
✓ 훈련 데이터: (256351, 69)
✓ 테스트 데이터: (90067, 68)

타겟 분포:
  실패 (0): 190,123
  성공 (1): 66,228


In [4]:
#전처리

print("\n" + "=" * 80)
print("🔧 전처리")
print("=" * 80)

# 변수 분류
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# 훈련 통계 추출
numeric_stats = {col: X_train[col].median() for col in numeric_cols}
categorical_stats = {col: X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else '알 수 없음'
                     for col in categorical_cols}

# 결측값 처리
def preprocess_data(df, numeric_stats, categorical_stats):
    df = df.copy()
    for col in categorical_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(categorical_stats[col])
    for col in numeric_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(numeric_stats[col])
    return df

X_train = preprocess_data(X_train, numeric_stats, categorical_stats)
X_test = preprocess_data(X_test, numeric_stats, categorical_stats)

# 로그 변환
high_skew_cols = ['저장된_배아_수', '해동된_배아_수', '저장된_신선_난자_수', '기증자_정자와_혼합된_난자_수']
for col in high_skew_cols:
    if col in X_train.columns:
        X_train[f'{col}_log'] = np.log1p(X_train[col])
        X_test[f'{col}_log'] = np.log1p(X_test[col])

print("✓ 결측값 처리 및 로그 변환 완료")


🔧 전처리
✓ 결측값 처리 및 로그 변환 완료


In [5]:
# 파생변수 30개 생성
print("\n" + "=" * 80)
print("🔬 파생변수 30개 생성")
print("=" * 80)

# --- Start of fix ---
# 특정 컬럼들의 데이터 타입을 숫자형으로 변환 (산술 연산 전에 필요)
# '시술 당시 나이' 컬럼 처리
if '시술 당시 나이' in X_train.columns:
    # '만'과 '세' 제거 후 첫 번째 숫자 추출 (예: '만18-34세' -> 18)
    X_train['시술 당시 나이'] = X_train['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)
    X_test['시술 당시 나이'] = X_test['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)

# '횟수'를 포함하는 컬럼 처리 (예: '0회' -> 0)
count_cols = [
    '총_임신_횟수', '총_시술_횟수', '총_출산_횟수',
    'IVF 임신 횟수', 'IVF 시술 횟수', 'DI 임신 횟수', 'DI 시술 횟수'
]

for col in count_cols:
    if col in X_train.columns and X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype(str).str.replace('회', '', regex=False)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    if col in X_test.columns and X_test[col].dtype == 'object':
        X_test[col] = X_test[col].astype(str).str.replace('회', '', regex=False)
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# 변환 후 발생할 수 있는 NaN 값은 마지막 fillna(0)에서 처리됨

print("✓ 특정 컬럼 숫자형 변환 완료")
# --- End of fix ---

# Group 1: 성공률
if '총_임신_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_성공_비율'] = X_train['총_임신_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_성공_비율'] = X_test['총_임신_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_성공_비율")

if '총_출산_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_출산_비율'] = X_train['총_출산_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_출산_비율'] = X_test['총_출산_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_출산_비율")

if '총_임신_횟수' in X_train.columns and '총_출산_횟수' in X_train.columns:
    X_train['유산율'] = 1 - (X_train['총_출산_횟수'] / (X_train['총_임신_횟수'] + 1))
    X_test['유산율'] = 1 - (X_test['총_출산_횟수'] / (X_test['총_임신_횟수'] + 1))
    X_train['유산율'] = X_train['유산율'].clip(0, 1)
    X_test['유산율'] = X_test['유산율'].clip(0, 1)
    print("✓ 유산율")

if 'IVF 임신 횟수' in X_train.columns and 'IVF 시술 횟수' in X_train.columns:
    X_train['IVF_성공_비율'] = X_train['IVF 임신 횟수'] / (X_train['IVF 시술 횟수'] + 1)
    X_test['IVF_성공_비율'] = X_test['IVF 임신 횟수'] / (X_test['IVF 시술 횟수'] + 1)
    print("✓ IVF_성공_비율")

if 'DI 임신 횟수' in X_train.columns and 'DI 시술 횟수' in X_train.columns:
    X_train['DI_성공_비율'] = X_train['DI 임신 횟수'] / (X_train['DI 시술 횟수'] + 1)
    X_test['DI_성공_비율'] = X_test['DI 임신 횟수'] / (X_test['DI 시술 횟수'] + 1)
    print("✓ DI_성공_비율")

# Group 2: 배아
if '총_생성_배아_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['배아_생성_효율'] = X_train['총_생성_배아_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['배아_생성_효율'] = X_test['총_생성_배아_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 배아_생성_효율")

if '이식된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_이식_효율'] = X_train['이식된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_이식_효율'] = X_test['이식된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_이식_효율")

if '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_저장률'] = X_train['저장된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_저장률'] = X_test['저장된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_저장률")

if '총_생성_배아_수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['배아_평균_생성_수'] = X_train['총_생성_배아_수'] / (X_train['총_시술_횟수'] + 1)
    X_test['배아_평균_생성_수'] = X_test['총_생성_배아_수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 배아_평균_생성_수")

if '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_다양성_지수'] = (X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_다양성_지수'] = (X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_다양성_지수")

if '총_생성_배아_수' in X_train.columns and '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns:
    X_train['배아_손실률'] = 1 - ((X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1))
    X_test['배아_손실률'] = 1 - ((X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1))
    X_train['배아_손실률'] = X_train['배아_손실률'].clip(0, 1)
    X_test['배아_손실률'] = X_test['배아_손실률'].clip(0, 1)
    print("✓ 배아_손실률")

# Group 3-8: 나머지 파생변수들
# (이전 파일의 코드를 여기 추가)

# 난자
if '수집된_신선_난자_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['신선_난자_비율'] = X_train['수집된_신선_난자_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['신선_난자_비율'] = X_test['수집된_신선_난자_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 신선_난자_비율")

if '시술 당시 나이' in X_train.columns:
    X_train['고령_여부'] = (X_train['시술 당시 나이'] >= 40).astype(int)
    X_test['고령_여부'] = (X_test['시술 당시 나이'] >= 40).astype(int)
    X_train['젊은_여부'] = (X_train['시술 당시 나이'] < 35).astype(int)
    X_test['젊은_여부'] = (X_test['시술 당시 나이'] < 35).astype(int)
    X_train['나이_제곱'] = X_train['시술 당시 나이'] ** 2
    X_test['나이_제곱'] = X_test['시술 당시 나이'] ** 2
    print("✓ 고령_여부, 젊은_여부, 나이_제곱")

if '총_시술_횟수' in X_train.columns:
    X_train['재시술_여부'] = (X_train['총_시술_횟수'] > 0).astype(int)
    X_test['재시술_여부'] = (X_test['총_시술_횟수'] > 0).astype(int)
    X_train['다중시술_여부'] = (X_train['총_시술_횟수'] >= 3).astype(int)
    X_test['다중시술_여부'] = (X_test['총_시술_횟수'] >= 3).astype(int)
    print("✓ 재시술_여부, 다중시술_여부")

# 결측값 처리
X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"\n✅ 파생변수 생성 완료! ({X_train.shape[1]} 특성)")


🔬 파생변수 30개 생성
✓ 특정 컬럼 숫자형 변환 완료
✓ IVF_성공_비율
✓ DI_성공_비율
✓ 고령_여부, 젊은_여부, 나이_제곱

✅ 파생변수 생성 완료! (72 특성)


In [6]:
# 범주형 인코딩

print("\n" + "=" * 80)
print("🔤 범주형 변수 인코딩")
print("=" * 80)

categorical_features = X_train.select_dtypes(include='object').columns.tolist()

for col in categorical_features:
    unique_categories = sorted(X_train[col].unique())
    category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
    X_train[col] = X_train[col].map(category_mapping)
    X_test[col] = X_test[col].map(category_mapping).fillna(-1).astype(int)

print(f"✓ {len(categorical_features)}개 변수 인코딩 완료")


🔤 범주형 변수 인코딩
✓ 15개 변수 인코딩 완료


In [ ]:
#고급 모델 학습

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("🤖 고급 머신러닝 & 신경망 모델 학습")
print("=" * 80)

# ============================================================================
# Step 1: 데이터 분할 (Stratified K-Fold)
# ============================================================================

print("\n【 Stratified K-Fold 교차 검증 】")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 모델별 예측값 저장
predictions = {
    'lgb': np.zeros(len(X_test)),
    'xgb': np.zeros(len(X_test)),
    'catboost': np.zeros(len(X_test)),
    'hgb': np.zeros(len(X_test)),
    'neural': np.zeros(len(X_test)),
}

cv_scores = {
    'lgb': [], 'xgb': [], 'catboost': [], 'hgb': [], 'neural': []
}

# ============================================================================
# Step 2: Fold별 학습
# ============================================================================

fold = 0
for train_idx, val_idx in skf.split(X_train, y_train):
    fold += 1
    print(f"\n【 Fold {fold}/5 】")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # ============================================================================
    # Model 1: LightGBM
    # ============================================================================

    print(f"  LightGBM 학습 중...")
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'num_leaves': 31,
        'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 20,
        'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
        'verbose': -1, 'scale_pos_weight': 2.87, 'seed': 42 + fold,
    }
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    lgb_model = lgb.train(lgb_params, train_data, num_boost_round=200,
                          valid_sets=[val_data],
                          callbacks=[lgb.log_evaluation(period=-1),
                                    lgb.early_stopping(stopping_rounds=30)])

    y_val_lgb = lgb_model.predict(X_val)
    y_test_lgb = lgb_model.predict(X_test)
    auc_lgb = roc_auc_score(y_val, y_val_lgb)
    cv_scores['lgb'].append(auc_lgb)
    predictions['lgb'] += y_test_lgb / 5
    print(f"    LightGBM AUC: {auc_lgb:.4f}")

    # ============================================================================
    # Model 2: XGBoost
    # ============================================================================

    print(f"  XGBoost 학습 중...")
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 6,
        'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8,
        'scale_pos_weight': 2.87, 'random_state': 42 + fold, 'tree_method': 'hist',
    }
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dtest = xgb.DMatrix(X_test)
    xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=200,
                         evals=[(dtrain, 'train'), (dval, 'eval')],
                         verbose_eval=False, early_stopping_rounds=30)

    y_val_xgb = xgb_model.predict(dval)
    y_test_xgb = xgb_model.predict(dtest)
    auc_xgb = roc_auc_score(y_val, y_val_xgb)
    cv_scores['xgb'].append(auc_xgb)
    predictions['xgb'] += y_test_xgb / 5
    print(f"    XGBoost AUC: {auc_xgb:.4f}")

    # ============================================================================
    # Model 3: CatBoost
    # ============================================================================

    print(f"  CatBoost 학습 중...")
    cb_model = cb.CatBoostClassifier(
        iterations=200, learning_rate=0.1, depth=6, verbose=False,
        scale_pos_weight=2.87, random_state=42 + fold,
        early_stopping_rounds=30
    )
    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

    y_val_cb = cb_model.predict_proba(X_val)[:, 1]
    y_test_cb = cb_model.predict_proba(X_test)[:, 1]
    auc_cb = roc_auc_score(y_val, y_val_cb)
    cv_scores['catboost'].append(auc_cb)
    predictions['catboost'] += y_test_cb / 5
    print(f"    CatBoost AUC: {auc_cb:.4f}")

    # ============================================================================
    # Model 4: HistGradientBoosting
    # ============================================================================

    print(f"  HistGradientBoosting 학습 중...")
    hgb_model = HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.1, max_depth=7,
        random_state=42 + fold, early_stopping=True,
        validation_fraction=0.2
    )
    hgb_model.fit(X_tr, y_tr)

    y_val_hgb = hgb_model.predict_proba(X_val)[:, 1]
    y_test_hgb = hgb_model.predict_proba(X_test)[:, 1]
    auc_hgb = roc_auc_score(y_val, y_val_hgb)
    cv_scores['hgb'].append(auc_hgb)
    predictions['hgb'] += y_test_hgb / 5
    print(f"    HistGradientBoosting AUC: {auc_hgb:.4f}")

    # ============================================================================
    # Model 5: Neural Network (DNN)
    # ============================================================================

    print(f"  Neural Network 학습 중...")

    # 데이터 스케일링
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    # 신경망 구축
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),

        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['auc']
    )

    # 클래스 가중치 (불균형 처리)
    class_weight = {0: 1.0, 1: 2.87}

    # 학습
    history = model.fit(
        X_tr_scaled, y_tr,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=32,
        class_weight=class_weight,
        verbose=0,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor='val_auc', mode='max', patience=15, restore_best_weights=True
            )
        ]
    )

    y_val_nn = model.predict(X_val_scaled, verbose=0).flatten()
    y_test_nn = model.predict(X_test_scaled, verbose=0).flatten()
    auc_nn = roc_auc_score(y_val, y_val_nn)
    cv_scores['neural'].append(auc_nn)
    predictions['neural'] += y_test_nn / 5
    print(f"    Neural Network AUC: {auc_nn:.4f}")

    # 메모리 정리
    keras.backend.clear_session()

# ============================================================================
# Step 3: 교차 검증 결과
# ============================================================================

print("\n" + "=" * 80)
print("【 교차 검증 결과 】")
print("=" * 80)

for model_name, scores in cv_scores.items():
    mean_auc = np.mean(scores)
    std_auc = np.std(scores)
    print(f"{model_name:15s}: {mean_auc:.4f} ± {std_auc:.4f}")

# ============================================================================
# Step 4: 특성 중요도 확인
# ============================================================================

print("\n" + "=" * 80)
print("【 상위 20개 중요 특성 (LightGBM) 】")
print("=" * 80)

feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': lgb_model.feature_importance()
}).sort_values('importance', ascending=False)

print(feature_importance.head(20).to_string(index=False))

top_20_features = feature_importance['feature'].head(20).tolist()
print(f"\n상위 20개 특성: {top_20_features[:5]}... (총 {len(top_20_features)}개)")

# ============================================================================
# Step 5: 최종 앙상블 (가중치 기반)
# ============================================================================

print("\n" + "=" * 80)
print("【 최종 앙상블 (가중치 기반) 】")
print("=" * 80)

# 교차검증 점수 기반 가중치
weights = {}
total_auc = sum(np.mean(scores) for scores in cv_scores.values())
for model_name, scores in cv_scores.items():
    mean_auc = np.mean(scores)
    weight = mean_auc / total_auc
    weights[model_name] = weight
    print(f"{model_name:15s}: {weight:.4f} (AUC: {mean_auc:.4f})")

# 가중치 적용
y_test_ensemble = (
    predictions['lgb'] * weights['lgb'] +
    predictions['xgb'] * weights['xgb'] +
    predictions['catboost'] * weights['catboost'] +
    predictions['hgb'] * weights['hgb'] +
    predictions['neural'] * weights['neural']
)

print(f"\n최종 앙상블:")
print(f"  예측값 범위: [{y_test_ensemble.min():.6f}, {y_test_ensemble.max():.6f}]")
print(f"  평균: {y_test_ensemble.mean():.6f}")

print("\n" + "=" * 80)
print("✅ 모델 학습 완료!")
print("=" * 80)

🤖 고급 머신러닝 & 신경망 모델 학습

【 Stratified K-Fold 교차 검증 】

【 Fold 1/5 】
  LightGBM 학습 중...
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[192]	valid_0's auc: 0.737455
    LightGBM AUC: 0.7375
  XGBoost 학습 중...
    XGBoost AUC: 0.7368
  CatBoost 학습 중...
    CatBoost AUC: 0.7379
  HistGradientBoosting 학습 중...
    HistGradientBoosting AUC: 0.7363
  Neural Network 학습 중...


In [ ]:
#예측 & 제출

import pandas as pd

print("\n" + "=" * 80)
print("📤 최종 예측 & 제출")
print("=" * 80)

# 제출 파일
submission = pd.DataFrame({
    'ID': test['ID'],
    'probability': y_test_ensemble
})

# 통계
print(f"\n예측값 통계:")
print(f"  최소: {submission['probability'].min():.6f}")
print(f"  최대: {submission['probability'].max():.6f}")
print(f"  평균: {submission['probability'].mean():.6f}")
print(f"  중앙값: {submission['probability'].median():.6f}")

# 저장
submission.to_csv('submission_04.24_2.csv', index=False)
submission.to_csv('/content/drive/MyDrive/submission_04.24_2.csv', index=False)

print(f"\n✓ submission_04.24_2.csv 생성 완료")
print(f"  샘플: {len(submission):,}")

# 다운로드
from google.colab import files
files.download('submission_04.24_2.csv')

print("\n" + "=" * 80)
print("✅ 모든 작업 완료!")
print("=" * 80)

print("\n【 최종 모델 구성 】")
print("✓ LightGBM (5-Fold CV)")
print("✓ XGBoost (5-Fold CV)")
print("✓ CatBoost (5-Fold CV)")
print("✓ HistGradientBoosting (5-Fold CV)")
print("✓ Neural Network (5-Fold CV)")
print("✓ 가중치 기반 앙상블")

In [ ]:
## 📌 **Cell 8: 모델 비교 시각화**

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. 모델별 CV 점수
ax = axes[0, 0]
model_names = list(cv_scores.keys())
mean_scores = [np.mean(scores) for scores in cv_scores.values()]
std_scores = [np.std(scores) for scores in cv_scores.values()]

colors = ['#4ECDC4', '#FFD93D', '#FF6B6B', '#95E1D3', '#C780FA']
ax.bar(model_names, mean_scores, yerr=std_scores, capsize=5, color=colors, edgecolor='black')
ax.set_ylabel('AUC-ROC')
ax.set_title('모델별 교차 검증 점수', fontsize=12, fontweight='bold')
ax.set_ylim([0.72, 0.74])
for i, v in enumerate(mean_scores):
    ax.text(i, v + 0.0005, f'{v:.4f}', ha='center', fontweight='bold', fontsize=9)

# 2. 모델 가중치
ax = axes[0, 1]
weight_names = list(weights.keys())
weight_values = list(weights.values())
ax.pie(weight_values, labels=weight_names, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('앙상블 모델 가중치', fontsize=12, fontweight='bold')

# 3. Fold별 성능
ax = axes[1, 0]
folds = np.arange(1, 6)
for model_name, scores in cv_scores.items():
    ax.plot(folds, scores, marker='o', label=model_name, linewidth=2)
ax.set_xlabel('Fold')
ax.set_ylabel('AUC-ROC')
ax.set_title('Fold별 모델 성능 추이', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. 최종 예측값 분포
ax = axes[1, 1]
ax.hist(y_test_ensemble, bins=50, color='#4ECDC4', edgecolor='black', alpha=0.7)
ax.axvline(y_test_ensemble.mean(), color='red', linestyle='--', linewidth=2,
           label=f'평균: {y_test_ensemble.mean():.4f}')
ax.set_xlabel('예측 확률')
ax.set_ylabel('빈도')
ax.set_title('최종 앙상블 예측값 분포', fontsize=12, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ model_comparison.png 저장 완료")